# Recommendation Engine - v1

Given a user's skills, recommend the most relevant postings and identify commonly-required skills the user is missing (skill gap).

**Approach: content-based filtering over skill vectors, IDF-weighted.** Each posting is represented as a binary vector over the 133 gazetteer skills (from `05_skill_extraction.ipynb`). A user profile is the same kind of vector. Recommendations are the postings with the highest cosine similarity to the user's vector.

**Why IDF weighting, not plain overlap:** the skill-extraction notebook's top-skills table showed soft skills (Leadership, Communication, Sales, Customer Service) mentioned in tens of thousands of postings, while specific technical/domain skills (Python, AutoCAD, HIPAA) are much rarer. Unweighted overlap would let two postings match mainly because they both mention "Leadership," drowning out more meaningful, specific overlaps. Weighting each skill by inverse document frequency (rare skills count for more) is a standard, defensible fix - this is the same idea as TF-IDF in text retrieval, applied to a skill vector instead of a word vector.

**Why content-based, not collaborative filtering:** we have no user interaction history (clicks, applications) to build a collaborative filter from - only posting content. Content-based is the only approach the data actually supports for v1.

No ground-truth labels exist for "correct" recommendations (flagged as an open evaluation question in the report outline), so validation here is: (1) manual sanity-checking against a few realistic user profiles, and (2) a simple quantitative comparison against a random-postings baseline.

## Setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

BASE = "./"
postings = pd.read_parquet(f"{BASE}postings_clean.parquet")
extracted = pd.read_parquet(f"{BASE}job_skills_extracted.parquet")
print(f"postings: {postings.shape}, extracted skill pairs: {extracted.shape}")

postings: (123849, 32), extracted skill pairs: (383288, 3)


## 1. Build the job x skill matrix

Dense binary matrix, `(n_postings, n_skills)`. Only 133 skill columns, so dense is simpler and fast enough - no need for sparse matrix machinery at this scale (~66MB as float32).

In [2]:
job_ids = postings["job_id"].values
job_id_to_idx = {jid: i for i, jid in enumerate(job_ids)}

skill_list = sorted(extracted["skill"].unique())
skill_to_idx = {s: i for i, s in enumerate(skill_list)}
skill_category = extracted.drop_duplicates("skill").set_index("skill")["category"].to_dict()

n_jobs, n_skills = len(job_ids), len(skill_list)
skill_matrix = np.zeros((n_jobs, n_skills), dtype=np.float32)

row_idx = extracted["job_id"].map(job_id_to_idx).values
col_idx = extracted["skill"].map(skill_to_idx).values
skill_matrix[row_idx, col_idx] = 1.0

print(f"skill_matrix: {skill_matrix.shape}, {skill_matrix.nbytes / 1e6:.1f} MB")

skill_matrix: (123849, 133), 65.9 MB


## 2. IDF weighting

`idf(skill) = log(n_postings / postings_containing_skill)`. Common skills (Leadership, appears in ~29K postings) get a low weight; rare skills (e.g. HIPAA) get a high weight.

In [3]:
doc_freq = skill_matrix.sum(axis=0)
idf = np.log(n_jobs / (doc_freq + 1))
weighted_matrix = skill_matrix * idf

idf_table = pd.DataFrame({"skill": skill_list, "postings_mentioning": doc_freq.astype(int), "idf_weight": idf})
print("lowest-weighted (most common) skills:")
print(idf_table.sort_values("idf_weight").head(5).to_string(index=False))
print("\nhighest-weighted (rarest) skills:")
print(idf_table.sort_values("idf_weight", ascending=False).head(5).to_string(index=False))

lowest-weighted (most common) skills:
           skill  postings_mentioning  idf_weight
      Leadership                29355    1.439566
   Communication                29034    1.450561
           Sales                28777    1.459452
Customer Service                24985    1.600747
 Problem Solving                19182    1.865039

highest-weighted (rarest) skills:
         skill  postings_mentioning  idf_weight
             R                   33    8.200458
         Flask                   81    7.320099
         NumPy                   85    7.272471
Medical Coding                   92    7.194219
          SPSS                   94    7.172942


## 3. Recommendation function

Cosine similarity between the user's weighted skill vector and every posting's weighted skill vector. Skills the user lists that aren't in our gazetteer are reported separately (`unmatched`) rather than silently dropped, so the limitation is visible.

In [4]:
matrix_norms = np.linalg.norm(weighted_matrix, axis=1)

def recommend(user_skills, top_n=10):
    matched = [s for s in user_skills if s in skill_to_idx]
    unmatched = [s for s in user_skills if s not in skill_to_idx]

    user_vec = np.zeros(n_skills, dtype=np.float32)
    for s in matched:
        user_vec[skill_to_idx[s]] = 1.0
    user_weighted = user_vec * idf
    user_norm = np.linalg.norm(user_weighted)

    scores = weighted_matrix @ user_weighted
    denom = matrix_norms * user_norm
    denom[denom == 0] = 1  # postings/user with zero matched skills -> similarity 0, not div-by-zero
    cosine = scores / denom

    top_idx = np.argsort(-cosine)[:top_n]
    result = postings.iloc[top_idx][["job_id", "title", "company_name", "location"]].copy()
    result["similarity"] = cosine[top_idx]
    return result.reset_index(drop=True), unmatched, top_idx

def skill_gaps(user_skills, top_idx, n_gaps=10):
    user_set = set(user_skills)
    sub_freq = skill_matrix[top_idx].sum(axis=0)
    gaps = [(skill_list[i], int(sub_freq[i])) for i in range(n_skills)
            if skill_list[i] not in user_set and sub_freq[i] > 0]
    gaps.sort(key=lambda x: -x[1])
    return gaps[:n_gaps]

print("recommend() and skill_gaps() defined")

recommend() and skill_gaps() defined


## 4. Manual validation against realistic profiles

Three profiles spanning different domains present in the dataset. Checking the recommended titles look plausible before trusting the numbers below.

In [5]:
test_profiles = {
    "Data Analyst": ["Python", "SQL", "Excel", "Data Analysis"],
    "Nurse": ["Nursing", "Patient Care", "HIPAA"],
    "Marketing": ["Marketing", "Digital Marketing", "SEO"],
}

for name, skills in test_profiles.items():
    recs, unmatched, top_idx = recommend(skills, top_n=10)
    gaps = skill_gaps(skills, top_idx, n_gaps=5)
    print(f"=== {name}: {skills} ===")
    if unmatched:
        print(f"(not in gazetteer, ignored: {unmatched})")
    print(recs[["title", "company_name", "similarity"]].to_string(index=False))
    print(f"top missing skills among these recommendations: {[g[0] for g in gaps]}")
    print()

=== Data Analyst: ['Python', 'SQL', 'Excel', 'Data Analysis'] ===
                            title                          company_name  similarity
     Corporate Compliance Officer                               Avalara    1.000000
                 Business Analyst                                 GTTSi    1.000000
           Data Analyst (Only W2)                Systellar Technologies    1.000000
                     Data Analyst                    Amtex Systems Inc.    1.000000
                     Data Analyst                  Gravity IT Resources    1.000000
                     Data Analyst                     SNVA Technologies    1.000000
        Senior Operations Analyst Mathys+Potestio / The Creative Party®    0.908243
              Senior Data Analyst                          Flexton Inc.    0.892448
Medical Data Analyst / Programmer                  Nippon Life Benefits    0.881919
          Data Privacy Specialist                        Selby Jennings    0.874488
top missin

**Check the output above.** Recommended titles should plausibly relate to each profile, and skill gaps should be sensible additions (e.g. a Data Analyst profile missing "Tableau" or "Statistics" makes sense; if gaps look random, revisit the approach before trusting the evaluation below.

## 5. Quantitative check against a random baseline

No ground-truth "correct recommendation" labels exist for this dataset, so we can't compute precision/recall. What we *can* check: are our recommendations meaningfully more similar to the user profile than a random set of postings would be? If not, the ranking isn't doing anything useful.

In [6]:
rng = np.random.default_rng(42)

eval_rows = []
for name, skills in test_profiles.items():
    recs, _, top_idx = recommend(skills, top_n=10)
    recommended_similarity = recs["similarity"].mean()

    random_idx = rng.choice(n_jobs, size=10, replace=False)
    user_vec = np.zeros(n_skills, dtype=np.float32)
    for s in skills:
        if s in skill_to_idx:
            user_vec[skill_to_idx[s]] = 1.0
    user_weighted = user_vec * idf
    user_norm = np.linalg.norm(user_weighted)
    random_scores = weighted_matrix[random_idx] @ user_weighted
    random_norms = matrix_norms[random_idx] * user_norm
    random_norms[random_norms == 0] = 1
    random_similarity = (random_scores / random_norms).mean()

    eval_rows.append({
        "profile": name, "top10_avg_similarity": recommended_similarity,
        "random10_avg_similarity": random_similarity,
        "lift": recommended_similarity / max(random_similarity, 1e-9),
    })

pd.DataFrame(eval_rows)

,profile,top10_avg_similarity,random10_avg_similarity,lift
0,Data Analyst,0.95571,0.025921,36.869476
1,Nurse,1.00000,0.032559,30.713703
2,Marketing,1.00000,0.060620,16.496305


## 6. Save the artifacts the dashboard will need

In [7]:
np.savez_compressed(
    f"{BASE}recommendation_artifacts.npz",
    weighted_matrix=weighted_matrix, idf=idf, matrix_norms=matrix_norms,
    job_ids=job_ids, skill_list=np.array(skill_list, dtype=object),
)
print("saved recommendation_artifacts.npz")

saved recommendation_artifacts.npz


## 7. Known limitations and next steps

- **No ground truth:** the "lift over random" check confirms the ranking isn't arbitrary, but doesn't tell us the recommendations are actually *good* - that needs qualitative review (Section 4 above) and, ideally, real user feedback we don't have.
- **Cold start for out-of-gazetteer skills:** a user listing a skill our extractor doesn't know about (`unmatched` above) is silently ignored rather than contributing to the match. Worth surfacing this to the user in the actual product rather than hiding it.
- **No diversity control:** top-N recommendations could all be near-duplicate postings from the same company/title. Not addressed in v1.
- **Next steps:** build the dashboard on top of `recommendation_artifacts.npz`; consider re-running this evaluation once the gazetteer is expanded (Section 7 of `05_skill_extraction.ipynb`).